# Búsqueda de mutaciones en PDB (1BTL, 1KZN, 1RX2)

Este notebook analiza los encabezados de archivos PDB dados y extrae mutaciones reportadas en: 
- Líneas `SEQADV` (conflictos entre la secuencia depositada y la referencia UniProt/DB).
- `REMARK 450` (frecuente en 1BTL: declara mutaciones como `V84I`, `A184V`).
- Cualquier `REMARK` que contenga la palabra `MUTATION`.

Genera un DataFrame con: `pdb_id`, `source`, `chain`, `position`, `observed`, `reference`, `detail` y guarda `mutations_summary.csv`.

In [ ]:
import re
import pandas as pd
from pathlib import Path

# Mapeo 3-letter a 1-letter para la secuencia
RES3_TO1 = {
    'ALA':'A','ARG':'R','ASN':'N','ASP':'D','CYS':'C','GLN':'Q','GLU':'E','GLY':'G',
    'HIS':'H','ILE':'I','LEU':'L','LYS':'K','MET':'M','PHE':'F','PRO':'P','SER':'S',
    'THR':'T','TRP':'W','TYR':'Y','VAL':'V'
}

def read_pdb_header(pdb_path: Path):
    """Lee sólo el encabezado (antes de la primera línea ATOM/HETATM).
    Devuelve lista de líneas de encabezado.
    """
    header_lines = []
    with pdb_path.open('r', encoding='utf-8', errors='ignore') as f:
        for line in f:
            if line.startswith('ATOM') or line.startswith('HETATM'):
                break
            header_lines.append(line.rstrip('
'))
    return header_lines

def parse_seqadv(pdb_id: str, header_lines):
    items = []
    # Ejemplo: SEQADV 1RX2 ASP A   37  UNP  P0ABQ4    ASN    37 CONFLICT
    seqadv_re = re.compile(r'^SEQADV\s+\S+\s+([A-Z]{3})\s+([A-Za-z])\s+(\d+)\s+\S+\s+\S+\s+([A-Z]{3})\s+(\d+)\s+(.*)$')
    for ln in header_lines:
        if ln.startswith('SEQADV'):
            m = seqadv_re.match(ln)
            if m:
                obs3, chain, pos, ref3, refpos, detail = m.groups()
                obs1 = RES3_TO1.get(obs3, '?')
                ref1 = RES3_TO1.get(ref3, '?')
                items.append({
                    'pdb_id': pdb_id,
                    'source': 'SEQADV',
                    'chain': chain.strip(),
                    'position': int(pos),
                    'observed': f'{obs3}({obs1})',
                    'reference': f'{ref3}({ref1})',
                    'detail': detail.strip()
                })
    return items

def parse_remark_mutations(pdb_id: str, header_lines):
    items = []
    # Buscar patrones tipo X99Y en líneas REMARK (p.ej. V84I, A184V)
    pat = re.compile(r'([A-Z])(\d+)([A-Z])')
    for ln in header_lines:
        if ln.startswith('REMARK'):
            if ('MUTATION' in ln.upper()) or ('DIFFERS' in ln.upper() and 'BY' in ln.upper()):
                for m in pat.finditer(ln):
                    frm, pos, to = m.groups()
                    items.append({
                        'pdb_id': pdb_id,
                        'source': 'REMARK',
                        'chain': 'A',
                        'position': int(pos),
                        'observed': frm,
                        'reference': to,
                        'detail': ln.strip()
                    })
    return items

def collect_mutations(pdb_path: Path):
    pdb_id = pdb_path.stem.upper()
    header = read_pdb_header(pdb_path)
    muts = []
    muts.extend(parse_seqadv(pdb_id, header))
    muts.extend(parse_remark_mutations(pdb_id, header))
    return muts

files = [
    Path('1BTL.pdb'),
    Path('1KZN.pdb'),
    Path('1RX2.pdb')
]

all_items = []
for fp in files:
    if fp.exists():
        all_items.extend(collect_mutations(fp))
    else:
        all_items.append({
            'pdb_id': fp.stem.upper(),
            'source': 'ERROR',
            'chain': '',
            'position': None,
            'observed': '',
            'reference': '',
            'detail': f'Archivo no encontrado: {fp}'
        })

df = pd.DataFrame(all_items)
df


In [ ]:
# Guardar CSV con el resumen de mutaciones
out_path = Path('mutations_summary.csv')
df.to_csv(out_path, index=False)
print(f'✅ Guardado: {out_path.resolve()}')
display(df)